# Skill Memory vs. OCL Survey strategies

This notebook runs the **same five Skill Memory replicates used for the comparison benchmark** (seeds 0–4), then compares them with the existing OCL Survey results using the repository post-processing utilities.

The Skill Memory results are generated fresh by `experiments/run_skill_memory_replicates.py`. No old Skill Memory result CSV is consumed.


## 0. Setup and fresh Skill Memory replicates

The experiment cell runs:

```bash
python experiments/run_skill_memory_replicates.py
```

The runner executes `experiments/main.py` for Skill Memory with `experiment.seed=0,1,2,3,4`, matching the five ER seeds.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/kobros-tech/ocl_survey.git"
REPO_REF = "feature/skill-memory-comparison-notebook"
COLAB_ROOT = Path("/content/ocl_survey")

if "google.colab" in sys.modules and not (COLAB_ROOT / "experiments" / "main.py").exists():
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, REPO_URL, str(COLAB_ROOT)],
        check=True,
    )

REPO_ROOT = COLAB_ROOT if (COLAB_ROOT / "experiments" / "main.py").exists() else Path.cwd().resolve()
if not (REPO_ROOT / "experiments" / "main.py").exists():
    for parent in [REPO_ROOT, *REPO_ROOT.parents]:
        if (parent / "experiments" / "main.py").exists():
            REPO_ROOT = parent
            break

if not (REPO_ROOT / "experiments" / "run_skill_memory_replicates.py").exists():
    raise FileNotFoundError(
        f"Could not find experiments/run_skill_memory_replicates.py under {REPO_ROOT}"
    )

print(f"Repository: {REPO_ROOT}")
print(f"Reference: {REPO_REF if 'google.colab' in sys.modules else 'local checkout'}")


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_ROOT)

subprocess.run(
    [sys.executable, "experiments/run_skill_memory_replicates.py"],
    cwd=REPO_ROOT,
    env=env,
    check=True,
)


## 1. Load results

The comparison requires Skill Memory and ER to have the same seed set: **0–4**. The notebook fails rather than silently comparing unequal replicate counts.


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.toolkit.process_results import extract_results
from src.toolkit.post_metrics import (
    compute_average_forgetting,
    compute_AAA,
    compute_wcacc,
)

RESULTS_ROOT = REPO_ROOT / "results"
BENCHMARK = "split_cifar100"
NUM_TASKS = 20
MEMORY_SIZE = 2000
EXPECTED_SEEDS = [0, 1, 2, 3, 4]

STRATEGIES = {
    "Skill Memory": "skill_memory",
    "ER": "er",
    "ER-ACE": "er_ace",
    "DER++": "der",
    "MIR": "mir",
    "ER + LwF": "er_lwf",
    "RAR": "rar",
    "SCR": "scr",
    "AGEM": "agem",
    "MER": "mer",
    "iCaRL": "icarl",
    "GDumb": "gdumb",
}

TEST_STREAM = "Top1_Acc_Stream/eval_phase/test_stream/Task000"
VALID_STREAM = "Top1_Acc_Stream/eval_phase/valid_stream/Task000"
TEST_EXP = "Top1_Acc_Exp/eval_phase/test_stream/Task000/Exp"

def result_path(prefix):
    return RESULTS_ROOT / f"{prefix}_{BENCHMARK}_{NUM_TASKS}_{MEMORY_SIZE}"

frames = {}
for label, prefix in STRATEGIES.items():
    path = result_path(prefix)
    if not path.is_dir():
        continue
    try:
        frames[label] = extract_results(str(path), verbose=False)
    except Exception as exc:
        warnings.warn(f"Could not load {label}: {exc}")

def seeds_for(frame):
    training = frame.get("training", pd.DataFrame())
    if "seed" not in training.columns:
        return []
    return sorted(training["seed"].dropna().astype(int).unique().tolist())

seed_table = pd.DataFrame([
    {"method": label, "seeds": seeds_for(frame)}
    for label, frame in frames.items()
])
display(seed_table)

for label in ("Skill Memory", "ER"):
    if label not in frames:
        raise RuntimeError(f"{label} results are required for the comparison.")
    actual = seeds_for(frames[label])
    if actual != EXPECTED_SEEDS:
        raise RuntimeError(
            f"{label} has seeds {actual}; expected exactly {EXPECTED_SEEDS}."
        )


## 2. Scalar comparison

For each seed, final accuracy and final average forgetting are computed from the same result stream. Forgetting uses the repository's corrected causal `max-past accuracy − current accuracy` implementation.

AAA and WC-Acc are shown only when the corresponding validation metrics are actually present in the result data.


In [ ]:
summary = []

for label, result in frames.items():
    df = result.get("training")
    if df is None or df.empty or TEST_STREAM not in df.columns:
        continue

    required = [f"{TEST_EXP}{i:03d}" for i in range(NUM_TASKS)]
    if not set(required).issubset(df.columns):
        warnings.warn(f"Skipping metrics for {label}: missing task accuracy columns")
        continue

    seed_values = []
    forgetting_values = []

    for seed, seed_df in df.groupby("seed"):
        seed_df = seed_df.sort_values("mb_index")
        final_row = seed_df.iloc[-1]
        seed_values.append(final_row[TEST_STREAM])

        forgetting_df = compute_average_forgetting(seed_df.copy(), NUM_TASKS)
        forgetting_values.append(forgetting_df.iloc[-1]["Average_Forgetting"])

    row = {
        "method": label,
        "n": len(seed_values),
        "final_accuracy": np.nanmean(seed_values),
        "final_accuracy_std": np.nanstd(seed_values, ddof=1) if len(seed_values) > 1 else np.nan,
        "forgetting": np.nanmean(forgetting_values),
        "forgetting_std": np.nanstd(forgetting_values, ddof=1) if len(forgetting_values) > 1 else np.nan,
        "AAA": np.nan,
        "AAA_std": np.nan,
        "WCAcc": np.nan,
        "WCAcc_std": np.nan,
    }

    try:
        aaa_df = compute_AAA(df.copy(), VALID_STREAM)
        aaa_values = [
            seed_df.sort_values("mb_index").iloc[-1]["AAA"]
            for _, seed_df in aaa_df.groupby("seed")
        ]
        if aaa_values:
            row["AAA"] = np.nanmean(aaa_values)
            row["AAA_std"] = np.nanstd(aaa_values, ddof=1) if len(aaa_values) > 1 else np.nan
    except Exception:
        pass

    try:
        wc_df = compute_wcacc(df.copy(), NUM_TASKS)
        wc_values = [
            seed_df.sort_values("mb_index").iloc[-1]["WCAcc"]
            for _, seed_df in wc_df.groupby("seed")
        ]
        if wc_values:
            row["WCAcc"] = np.nanmean(wc_values)
            row["WCAcc_std"] = np.nanstd(wc_values, ddof=1) if len(wc_values) > 1 else np.nan
    except Exception:
        pass

    summary.append(row)

summary_df = pd.DataFrame(summary).sort_values("final_accuracy", ascending=False)
display(summary_df.style.format({
    "final_accuracy": "{:.2%}",
    "final_accuracy_std": "{:.2%}",
    "forgetting": "{:.2%}",
    "forgetting_std": "{:.2%}",
    "AAA": "{:.2%}",
    "AAA_std": "{:.2%}",
    "WCAcc": "{:.2%}",
    "WCAcc_std": "{:.2%}",
}))


## 3. Online accuracy

Curves are aggregated across seeds. Skill Memory and ER therefore use the same five-replicate basis.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for label, result in frames.items():
    df = result.get("training")
    if df is None or df.empty or TEST_STREAM not in df.columns:
        continue

    x = (
        df[["seed", "mb_index", TEST_STREAM]]
        .dropna()
        .sort_values(["seed", "mb_index"])
        .copy()
    )
    g = x.groupby("mb_index")[TEST_STREAM].agg(["mean", "std"]).reset_index()

    ax.plot(g["mb_index"], 100 * g["mean"], label=label)
    if len(g) > 1:
        std = g["std"].fillna(0)
        ax.fill_between(
            g["mb_index"],
            100 * (g["mean"] - std),
            100 * (g["mean"] + std),
            alpha=0.10,
        )

ax.set(
    xlabel="Batch index",
    ylabel="Test-stream accuracy (%)",
    title=f"{BENCHMARK}: online accuracy",
)
ax.legend(ncol=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 4. Causal catastrophic forgetting

This plot uses exactly the same `compute_average_forgetting` calculation as the scalar forgetting column above. Lower is better.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

for label, result in frames.items():
    df = result.get("training")
    if df is None or df.empty:
        continue

    required = [f"{TEST_EXP}{i:03d}" for i in range(NUM_TASKS)]
    if not set(required).issubset(df.columns):
        continue

    forgetting_df = compute_average_forgetting(df.copy(), NUM_TASKS)
    x = forgetting_df[["seed", "mb_index", "Average_Forgetting"]].dropna()
    if x.empty:
        continue

    g = x.groupby("mb_index")["Average_Forgetting"].agg(["mean", "std"]).reset_index()
    ax.plot(g["mb_index"], 100 * g["mean"], label=label)

    if len(g) > 1:
        std = g["std"].fillna(0)
        ax.fill_between(
            g["mb_index"],
            100 * (g["mean"] - std),
            100 * (g["mean"] + std),
            alpha=0.10,
        )

ax.axhline(0, linewidth=1)
ax.set(
    xlabel="Batch index",
    ylabel="Average forgetting (%)",
    title=f"{BENCHMARK}: causal catastrophic forgetting over training",
)
ax.legend(ncol=2, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## 5. Save the comparison

The CSV is generated from the current five-seed results and corrected forgetting implementation. It is not a checked-in source of truth; rerun this notebook after regenerating results.


In [ ]:
analysis_dir = RESULTS_ROOT / "analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)
summary_path = analysis_dir / "skill_memory_strategy_comparison.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Wrote {summary_path}")
